# 03_review — delivery review

Checks evidence, metadata, ownership, rules, and readiness for outputs published by `02_pipeline`. Run this after `01_agreement` and `02_pipeline` as the final required delivery review step.

Required delivery flow: `01_agreement` → `02_pipeline` → `03_review`. Optional support lives in `99_explore`.

This notebook selects a logical table from `METADATA_DATA_CATALOGUE` and records approved governance metadata only after explicit human commit actions. Approved DQ expectations become active for future pipeline runs when `02_pipeline` loads `METADATA_DQ_RULES` through `enforce_dq_rules`; this review notebook does not enforce them directly.


## 1. Run `00_env_config`

In [ ]:
%run 00_env_config


## 2. Import supported public APIs

In [ ]:
from fabricops_kit import (
    get_selected_catalogue_table,
    load_catalogue_profile_rows,
    record_table_governance,
    widget_review_column_classification,
    widget_review_column_context,
    widget_review_dq_rules,
    widget_select_catalogue_table,
)


## 3. Select catalogue table

In [ ]:
env_name = ENV

# Reads METADATA_DATA_CATALOGUE from the configured metadata target and selects
# one logical table using the latest successful profile run.
table_selector = widget_select_catalogue_table(CONFIG, env_name, spark_session=spark)
selected_table = get_selected_catalogue_table(table_selector)
selected_table


## 4. Show selected profile summary

In [ ]:
profile_rows = load_catalogue_profile_rows(CONFIG, env_name, selected_table, spark_session=spark)
profile_summary = {**selected_table, "column_count": len(profile_rows)}
display(spark.createDataFrame([profile_summary]))
display(spark.createDataFrame(profile_rows))


## 5. Review business context

The review state below is intentionally non-persistent. Edit the `reviewed_context_rows` list, set `commit=True` only for approved rows, then pass the reviewed list to the consolidated commit cell. AI suggestions may be copied into `ai_suggestion_json`, accepted, ignored, or changed before commit.

In [ ]:
context_review = widget_review_column_context(profile_rows)

# Optional Fabric AI examples can be added locally, but human approval remains mandatory.
# Add reviewed rows to this list and set commit=True when ready to persist evidence.
reviewed_context_rows = context_review + [
    # {"column_name": "example_column", "business_context": "Human-approved meaning", "notes": "Reviewed in 03_review", "review_status": "approved", "commit": True}
]


## 6. Review DQ rules

Author rules manually or use AI suggestions as advisory drafts. Approved active DQ rules are stored in `METADATA_DQ_RULES`; they are enforced only when a later `02_pipeline` run calls `enforce_dq_rules` before the target write.


In [ ]:
dq_review = widget_review_dq_rules(profile_rows)
reviewed_dq_rules = dq_review + [
    # {"rule_id": "orders.order_id.not_null", "column_name": "order_id", "rule_type": "not_null", "rule_parameters": {}, "severity": "error", "description": "Human-approved not-null expectation", "review_status": "approved", "is_active": True, "commit": True}
]


## 7. Review sensitivity and PII classification

In [ ]:
classification_review = widget_review_column_classification(profile_rows)
reviewed_classification_rows = classification_review + [
    # {"column_name": "customer_id", "sensitivity_label": "confidential", "personal_data_classification": "indirect_identifier", "pii_identifier_type": "customer key", "handling_requirement": "Limit to approved business users", "reasoning": "Human-reviewed identifier column", "review_status": "approved", "commit": True}
]

governance_records = record_table_governance(
    CONFIG,
    env_name,
    profile_rows,
    spark_session=spark,
    context_reviews=reviewed_context_rows,
    dq_rule_reviews=reviewed_dq_rules,
    classification_reviews=reviewed_classification_rows,
)
print({name: len(rows) for name, rows in governance_records.items()})


## 8. Display governance completion summary

In [ ]:
completion_summary = {
    "environment_name": selected_table["environment_name"],
    "dataset_name": selected_table["dataset_name"],
    "table_name": selected_table["table_name"],
    "profile_run_id": selected_table["profile_run_id"],
    "business_context_rows_committed": len(governance_records["column_context"]),
    "dq_rule_rows_committed": len(governance_records["dq_rules"]),
    "classification_rows_committed": len(governance_records["column_classification"]),
    "dq_enforcement_scope": "Approved active DQ rules are read by 02_pipeline through enforce_dq_rules; warning severity continues and error severity blocks before target write.",
}
display(spark.createDataFrame([completion_summary]))


## Notebook-output examples for PR review

- **Business context stage:** selected profile columns display with existing approved context and manual edit instructions.
- **DQ stage:** approved rules are append-only metadata events in `METADATA_DQ_RULES`; `02_pipeline` loads active approved rules with `enforce_dq_rules` on a later run.
- **Classification stage:** sensitivity and personal-data labels require human commit; AI is optional and advisory.
